In [ ]:
import os
import sys
import json
EVAL_DIR = os.getcwd() 
PROJECT_ROOT = os.path.dirname(EVAL_DIR)

os.environ["HF_HOME"] = os.path.join(PROJECT_ROOT, ".cache", "huggingface")
os.environ["TORCH_HOME"] = os.path.join(PROJECT_ROOT, ".cache", "torch")

print("HF Cache Path:", os.environ["HF_HOME"])

# 2. Setup sys.path
for path in [PROJECT_ROOT, EVAL_DIR]:
    if path not in sys.path:
        sys.path.insert(0, path)

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from lavis.models import load_model_and_preprocess
from lavis.models.blip_models.blip_image_text_matching import compute_gradcam

# 4. Tối ưu hóa GPU
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
input_path = os.path.join(PROJECT_ROOT, "AMBER", "error_llava.json")
output_path = os.path.join(PROJECT_ROOT, "AMBER", "amber_generative.jsonl")

In [ ]:
try:
    with open(input_path, "r", encoding="utf-8") as f_in:
        data = json.load(f_in)
    
    with open(output_path, "w", encoding="utf-8") as f_out:
        for item in data:
            new_record = {
                "question_id": item["id"],
                "image": item["image"],
                "text": "Describe this image."
            }
            f_out.write(json.dumps(new_record, ensure_ascii=False) + "\n")
            
    print(f"✅ Chuyển đổi thành công! Đã lưu {len(data)} dòng vào: {output_path}")

except FileNotFoundError:
    print(f"❌ Không tìm thấy file gốc tại: {input_path}. Bạn kiểm tra lại đường dẫn nhé!")
except Exception as e:
    print(f"❌ Có lỗi xảy ra: {e}")

In [ ]:
_to_pil = transforms.ToPILImage()

def augmentation_notebook(image, question, tensor_image, model, tokenized_text):
    device = image.device

    # 1. Tính toán ITC Score
    with torch.no_grad():
        itc_score = model({"image": image, "text_input": question}, match_head="itc")
    ratio: float = (1.0 - itc_score / 2.0).clamp(max=1.0 - 1e-5).item()

    # 2. Lấy Grad-CAM
    with torch.set_grad_enabled(True):
        gradcams, _ = compute_gradcam(model=model, visual_input=image, text_input=question, 
                                      tokenized_text=tokenized_text, block_num=6)

    gradcams = [gradcam_[1] for gradcam_ in gradcams]
    gradcam = torch.stack(gradcams).reshape(image.size(0), -1).reshape(24, 24)

    # 3. Nội suy và làm mịn Grad-CAM
    gradcam_raw = gradcam.float().to(device=device)
    g_min, g_max = gradcam_raw.min(), gradcam_raw.max()
    gradcam_norm = (gradcam_raw - g_min) / (g_max - g_min + 1e-8)

    gradcam_up = F.interpolate(gradcam_norm.unsqueeze(0).unsqueeze(0), size=(384, 384), 
                               mode="bicubic", align_corners=False)
    
    avg_gradcam_gpu = TF.gaussian_blur(gradcam_up, kernel_size=[63, 63], sigma=[7.68, 7.68]).squeeze()

    # 4. Tạo Mask và áp dụng vào ảnh
    threshold = torch.quantile(avg_gradcam_gpu.reshape(-1), 1.0 - ratio)
    mask_hw  = (avg_gradcam_gpu >= threshold).to(dtype=torch.float32)          
    mask_hwc = mask_hw.unsqueeze(2).expand(-1, -1, 3)                          

    new_image_gpu = tensor_image.to(device=device).permute(1, 2, 0) * mask_hwc  

    # Convert ảnh về PIL và heatmap về Numpy
    imag = _to_pil(new_image_gpu.cpu().permute(2, 0, 1).clamp(0.0, 1.0))
    heatmap_np = avg_gradcam_gpu.cpu().numpy()
    
    return imag, heatmap_np

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
has_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
runtime_dtype = torch.bfloat16 if has_bf16 else torch.float16

print("⏳ Loading BLIP-ITM model...")
model_itm, vis_processors, text_processors = load_model_and_preprocess(
    name="blip_image_text_matching", 
    model_type="base", # Hoặc "base" tùy model bạn dùng
    is_eval=True, 
    device=device
)
model_itm.to(runtime_dtype)
model_itm.requires_grad_(False)
print("✅ Load model thành công!")

In [ ]:
# CẤU HÌNH ĐƯỜNG DẪN (Bạn sửa lại ở đây)
JSONL_FILE = output_path 
IMAGE_FOLDER = os.path.join(PROJECT_ROOT, "AMBER", "image")
MAX_VISUALIZE = 10

loader = transforms.Compose([transforms.ToTensor()])

with open(JSONL_FILE, "r", encoding="utf-8") as f:
    questions = [json.loads(line) for line in f if line.strip()]

for i, line in enumerate(questions[:MAX_VISUALIZE]):
    # 1. Đọc thông tin từ file
    image_file = line["image"]
    raw_query = line.get("query") or line.get("text") or ""
    image_path = os.path.join(IMAGE_FOLDER, image_file)
    
    if not os.path.exists(image_path):
        print(f"❌ Không tìm thấy ảnh: {image_path}")
        continue
        
    # 2. Tiền xử lý ảnh và text
    raw_image = Image.open(image_path).convert("RGB")
    tensor_image = loader(raw_image.resize((384, 384))).float()
    
    image_itm_input = vis_processors["eval"](raw_image).unsqueeze(0).to(device=device, dtype=runtime_dtype)
    image_itm_input.requires_grad_(True) # Bắt buộc để tính đạo hàm

    itm_text = text_processors["eval"](raw_query)
    tokenized_text = model_itm.tokenizer(itm_text, padding='longest', truncation=True, return_tensors="pt").to(device)

    # 3. Chạy inference
    augmented_img, heatmap = augmentation_notebook(
        image_itm_input, itm_text, tensor_image, model_itm, tokenized_text
    )
    
    # 4. TRỰC QUAN HÓA THẲNG LÊN NOTEBOOK
    bg_img = tensor_image.permute(1, 2, 0).cpu().numpy() # Dùng tensor image làm background cho khớp size 384x384
    
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Query: \"{raw_query}\"", fontsize=16, fontweight='bold')
    
    # Ảnh gốc
    axs[0].imshow(bg_img)
    axs[0].set_title("Original Image (384x384)")
    axs[0].axis('off')
    
    # Grad-CAM Heatmap
    axs[1].imshow(bg_img)
    axs[1].imshow(heatmap, cmap='jet', alpha=0.5) # Chồng heatmap lên ảnh
    axs[1].set_title("Grad-CAM Heatmap")
    axs[1].axis('off')
    
    # Ảnh Augmentation
    axs[2].imshow(augmented_img)
    axs[2].set_title("Augmented (Masked) Image")
    axs[2].axis('off')
    
    plt.tight_layout()
    plt.show() # Hiển thị trực tiếp, không lưu